In [1]:
import os
import json
import re
from pathlib import Path
from collections import defaultdict

import torch
from datasets import Dataset

from rapidfireai import Experiment
from rapidfireai.automl import List, RFGridSearch, RFModelConfig, RFLoraConfig, RFSFTConfig

In [2]:
PROJECT_DIR = Path(".")  # 项目根目录；默认 notebook 从当前项目目录运行
SCHEMAS_DIR = PROJECT_DIR / "schemas"  # Spider 格式数据库 schema 所在目录
TRAIN_PATH = PROJECT_DIR / "train.json"  # 训练集路径，包含 question、gold_sql 和 schema_links
VAL_PATH = PROJECT_DIR / "validation.json"  # 验证集路径，用于 RapidFireAI config 间比较

SYSTEM_PROMPT = """You are a schema linking model for text-to-SQL.
Given a natural language question and a database schema, return ONLY valid JSON.
The JSON format must be:
{"TableName": ["Column1", "Column2"], "AnotherTable": []}

Rules:
- Use only table and column names from the provided schema.
- Include a table with [] if the table is referenced but no specific column is needed.
- Do not explain.
- Do not output markdown.
"""  # 系统提示词：约束模型只输出符合评分格式的 schema_links JSON

In [3]:
def schema_file_for_db(db_id, schemas_dir=SCHEMAS_DIR):
    fname = db_id.replace(" ", "_").replace("/", "_") + ".json"  # 按 README 规则把 db_id 转成 schema 文件名
    return schemas_dir / fname  # 返回该数据库对应的 schema JSON 路径


def load_spider_schema(db_id, schemas_dir=SCHEMAS_DIR):
    with open(schema_file_for_db(db_id, schemas_dir)) as f:  # 打开目标数据库的 Spider schema 文件
        s = json.load(f)  # 读取 schema JSON

    table_names = s["table_names_original"]  # 使用原始表名，保证输出 casing 与 schema 一致
    columns_by_table = {t: [] for t in table_names}  # 初始化 {table: [columns]} 结构

    for tidx, cname in s["column_names_original"]:  # Spider 中每列是 [table_index, column_name]
        if tidx == -1:  # tidx=-1 是 synthetic '*'，不能当作真实列
            continue
        columns_by_table[table_names[tidx]].append(cname)  # 把列名放回所属表下面

    return columns_by_table  # 返回便于 prompt 和后处理使用的 schema 字典


def serialize_schema(db_id):
    schema = load_spider_schema(db_id)  # 加载当前问题对应数据库的 schema
    lines = [f"Database: {db_id}", "Schema:"]  # prompt 中先说明数据库 ID 和 schema 区块
    for table, cols in schema.items():  # 逐表序列化所有列名
        lines.append(f"- {table}({', '.join(cols)})")  # 用 Table(col1, col2, ...) 的紧凑格式放进 prompt
    return "\n".join(lines)  # 拼成多行文本，作为模型输入的一部分


def build_user_prompt(question, db_id):
    return f"""{serialize_schema(db_id)}

Question:
{question}

Return schema_links JSON only."""  # 用户提示词：schema + 自然语言问题 + 输出格式要求

In [ ]:
def load_records(path):
    with open(path) as f:
        return json.load(f)


train_records = load_records(TRAIN_PATH)
val_records = load_records(VAL_PATH)


def add_prompt_fields(records):
    out = []
    for row in records:
        new_row = dict(row)
        new_row["user_prompt"] = build_user_prompt(row["question"], row["db_id"])
        new_row["assistant_response"] = json.dumps(
            row["schema_links"],
            ensure_ascii=False,
            separators=(",", ":"),
        )
        out.append(new_row)
    return out

# #  get train and validation dataset
train_dataset = Dataset.from_list(add_prompt_fields(train_records)).shuffle(seed=42)
eval_dataset = Dataset.from_list(add_prompt_fields(val_records)).shuffle(seed=42)

In [5]:
def formatting_function(row):
    system_prompt = """You are a schema linking model for text-to-SQL.
Given a natural language question and a database schema, return ONLY valid JSON.
The JSON format must be:
{"TableName": ["Column1", "Column2"], "AnotherTable": []}

Rules:
- Use only table and column names from the provided schema.
- Include a table with [] if the table is referenced but no specific column is needed.
- Do not explain.
- Do not output markdown.
"""

    return {
        "prompt": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row["user_prompt"]},
        ],
        "completion": [
            {"role": "assistant", "content": row["assistant_response"]}
        ],
    }

In [ ]:
def extract_json_object(text):
    if not isinstance(text, str):  # 防止 prediction 不是字符串时 json.loads 报错
        return {}

    text = text.strip()  # 去掉模型输出首尾空白
    text = re.sub(r"^```(?:json)?", "", text).strip()  # 容错：去掉开头 markdown JSON 代码块标记
    text = re.sub(r"```$", "", text).strip()  # 容错：去掉结尾 markdown 代码块标记

    try:
        obj = json.loads(text)  # 优先尝试把完整输出解析成 JSON
        return obj if isinstance(obj, dict) else {}  # 评分要求 schema_links 是 dict，不接受 list/string
    except Exception:
        pass  # 如果完整解析失败，再尝试从文本中截取第一个 JSON 对象

    match = re.search(r"\{.*\}", text, flags=re.S)  # 容错：从带解释文字的输出中提取 {...}
    if not match:
        return {}

    try:
        obj = json.loads(match.group(0))  # 解析截取出来的 JSON 片段
        return obj if isinstance(obj, dict) else {}  # 只保留 dict 类型输出
    except Exception:
        return {}  # 解析失败时返回空预测，避免程序中断


In [ ]:
# # 模型输出postprocess，主要用于推理阶段
# # 把模型输出里的表名、列名按 schema 做大小写恢复; 删除 schema 中不存在的表名; 删除不属于该表的列名; 去重、排序; 保留 {"Table": []} 这种 table-only 输出。


# def canonicalize_schema_links(raw_links, db_id): # post processing
#     schema = load_spider_schema(db_id)  # 加载当前 db_id 的合法表列集合
#     table_map = {t.lower(): t for t in schema}  # 表名大小写归一化，输出时恢复原始 casing
#     col_maps = {
#         t: {c.lower(): c for c in cols}  # 每张表内列名大小写归一化，输出时恢复原始 casing
#         for t, cols in schema.items()
#     }

#     clean = {}  # 保存过滤后的合法 schema_links
#     if not isinstance(raw_links, dict):  # 如果模型输出不是 dict，直接返回空结果
#         return clean

#     for raw_t, raw_cols in raw_links.items():  # 遍历模型预测的表和列
#         t_key = str(raw_t).lower()  # 表名转小写用于匹配 schema
#         if t_key not in table_map:  # 过滤 hallucinated table，避免 precision 被扣分
#             continue

#         table = table_map[t_key]  # 恢复 schema 中的原始表名 casing
#         cols = []  # 当前表下保留下来的合法列
#         if isinstance(raw_cols, list):  # 只有 list 才作为列集合处理
#             for c in raw_cols:
#                 c_key = str(c).lower()  # 列名转小写用于匹配 schema
#                 if c_key in col_maps[table]:  # 只保留属于该表的合法列
#                     cols.append(col_maps[table][c_key])  # 恢复 schema 中的原始列名 casing

#         clean[table] = sorted(set(cols))  # 去重并排序，保证输出稳定；空列表表示 table-only link

#     return clean  # 返回可直接提交给 eval.py 的过滤后结果

In [ ]:
def schema_link_metrics(eval_preds):
    """
    RapidFire demo 中 compute_metrics 接收 predictions, labels。
    这里做一个轻量 JSON exact-ish 指标，主要用于 config 间比较。
    最终分数仍然用 eval.py。
    """
    predictions, labels = eval_preds  # RapidFireAI 传入的模型生成结果和参考答案文本

    valid_json = 0  # 统计模型输出能否解析出 JSON dict
    exact = 0  # 统计预测 JSON 与标签 JSON 是否完全一致
    total = len(predictions)  # 验证样本总数，用于计算比例

    for pred, label in zip(predictions, labels):  # 逐条比较预测和标签
        p = extract_json_object(pred)  # 从模型输出中提取 JSON
        g = extract_json_object(label)  # 从标签文本中提取 JSON

        if p:  # 非空 dict 说明模型至少输出了可解析 JSON
            valid_json += 1
        if p == g:  # exact match 是严格指标，只用于快速筛 config
            exact += 1

    return {
        "valid_json_rate": round(valid_json / max(total, 1), 4),  # JSON 合法率，越高说明输出格式越稳定
        "exact_json_rate": round(exact / max(total, 1), 4),  # 完全匹配率，辅助比较不同训练配置
    }

In [8]:
def create_model(model_config):
    from transformers import AutoModelForCausalLM, AutoTokenizer  # 在函数内导入，方便 RapidFireAI 按 config 创建模型

    model_name = model_config["model_name"]  # 当前 RapidFireAI config 指定的 base model 名称
    model_kwargs = model_config["model_kwargs"]  # 当前 config 指定的加载参数，如 device_map 和 dtype

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)  # 加载 causal LM base model
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)  # 加载对应 tokenizer，保证 chat template 匹配

    if tokenizer.pad_token is None:  # 某些 causal LM 没有单独 pad token
        tokenizer.pad_token = tokenizer.eos_token  # 用 eos 作为 pad，避免 batch padding 报错
    tokenizer.padding_side = "right"  # SFT 训练通常使用右 padding，更符合 TRL 默认习惯

    return model, tokenizer  # RapidFireAI 需要返回 (model, tokenizer) 元组

In [9]:
peft_configs = List([  # RapidFireAI 的 List 会把多个 LoRA 设置展开成多组实验
    RFLoraConfig(
        r=8,  # LoRA rank 较小，参数少、训练快
        lora_alpha=16,  # LoRA 缩放系数，通常设置为 r 的 2 倍左右
        lora_dropout=0.05,  # 轻微 dropout，降低小数据集过拟合风险
        target_modules=["q_proj", "v_proj"],  # 只训练 attention 的 Q/V 投影，作为轻量 baseline
        bias="none",  # 不训练 bias，减少可训练参数
    ),
    # RFLoraConfig(
    #     r=16,  # 更大的 LoRA rank，表达能力更强但训练更慢
    #     lora_alpha=32,  # 与 r=16 搭配的缩放系数
    #     lora_dropout=0.05,  # 保持同样 dropout，便于比较 rank/target_modules 的影响
    #     target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # 覆盖更多 attention 投影，适合更强配置
    #     bias="none",  # 继续不训练 bias，保持 PEFT 设置简单
    # ),
])

common_generation_config = {  # 各 config 共用的验证生成参数
    "max_new_tokens": 256,  # schema_links JSON 通常较短，限制长度可减少废话输出
    "do_sample": False,  # 使用贪心/确定性解码，保证验证结果稳定
    "temperature": None,  # 不采样时不需要 temperature
    "top_p": None,  # 不采样时不需要 nucleus sampling
    "repetition_penalty": 1.02,  # 轻微惩罚重复，减少重复列名/表名
}

common_model_kwargs = {  # 各 base model 共用的加载参数
    "device_map": "auto",  # 自动把模型放到可用 GPU/设备上
    "torch_dtype": "auto",  # 自动选择模型权重 dtype，通常会用 bf16/fp16
    "use_cache": False,  # 训练时关闭 KV cache，减少显存并避免梯度检查点相关问题
}

In [10]:
# 2 base models x 2 learning rates x 2 LoRA configs = 8 configs，满足项目要求的多配置实验
config_set = List([  # RapidFireAI 会对这里的 RFModelConfig 和内部 peft_configs 做 grid 展开
    RFModelConfig(
        model_name="Qwen/Qwen2.5-0.5B-Instruct",  # 0.5B 小模型，训练快，用作低成本 baseline
        peft_config=peft_configs,  # 复用上面定义的 2 组 LoRA 配置
        training_args=RFSFTConfig(  # TRL SFTTrainer 相关训练参数
            learning_rate=2e-4,  # 中等学习率，适合 LoRA SFT 起步
            lr_scheduler_type="linear",  # cosine # cosine decay，训练后期逐渐降低学习率
            per_device_train_batch_size=4,  
            per_device_eval_batch_size=4,  
            gradient_accumulation_steps=8,  # 累积 8 步，等效 batch size 约为 8
            num_train_epochs=4,  # 小模型多训练一轮，帮助学习 JSON 输出格式
            logging_steps=5,  # 每 5 步记录一次训练日志，便于观察 loss
            eval_strategy="steps",  # 按 step 做验证，而不是只在 epoch 末验证
            eval_steps=25,  # 每 25 步评估一次，方便 RapidFireAI 早期比较 config
            save_steps=25,  # 每 25 步保存 checkpoint，便于挑选较好模型
            bf16=True,  # DSMLP 新 GPU 支持 bf16，可节省显存并保持稳定
        ),
        model_type="causal_lm",  # Qwen Instruct 属于 causal language model
        model_kwargs=common_model_kwargs,  # 使用共用模型加载参数
        formatting_func=formatting_function,  # 把原始样本转成 chat-style SFT 样本
        compute_metrics=schema_link_metrics,  # # 验证时计算 eval.py 风格的 leaderboard/table/column 指标
        generation_config=common_generation_config,  # 验证生成时使用确定性 JSON 解码配置
    ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen2.5-0.5B-Instruct",  # 同一个 0.5B base model，单独测试更大学习率
    #     peft_config=peft_configs,  # 继续和 2 组 LoRA 配置组合
    #     training_args=RFSFTConfig(
    #         learning_rate=5e-4,  # 较大学习率，观察小模型是否能更快适配任务格式
    #         lr_scheduler_type="cosine",  # 保持 scheduler 不变，减少对比变量
    #         per_device_train_batch_size=1,  # 控制显存占用
    #         per_device_eval_batch_size=1,  # 控制验证显存占用
    #         gradient_accumulation_steps=8,  # 保持等效 batch size 一致
    #         num_train_epochs=4,  # 与上一组 0.5B 实验保持 epoch 一致
    #         logging_steps=5,  # 高频日志便于发现不稳定训练
    #         eval_strategy="steps",  # step-level validation 便于 config 比较
    #         eval_steps=25,  # 每 25 步验证
    #         save_steps=25,  # 每 25 步保存
    #         bf16=True,  # 使用 bf16 降低显存压力
    #     ),
    #     model_type="causal_lm",  # causal LM 训练模式
    #     model_kwargs=common_model_kwargs,  # 共用 device/dtype/cache 设置
    #     formatting_func=formatting_function,  # 共用数据格式化函数
    #     compute_metrics=schema_link_metrics,  # 共用验证指标
    #     generation_config=common_generation_config,  # 共用生成参数
    # ),
    RFModelConfig(
        model_name="Qwen/Qwen2.5-1.5B-Instruct",  # 1.5B 模型，容量更大但仍满足 <=2B 限制
        peft_config=peft_configs,  # 与同样 2 组 LoRA 配置组合
        training_args=RFSFTConfig(
            learning_rate=1e-4,  # 大模型用更保守学习率，降低过拟合/发散风险
            lr_scheduler_type="cosine",  # 与其他实验保持一致
            per_device_train_batch_size=1,  # 1.5B 显存占用更高，batch size 保守设置
            per_device_eval_batch_size=1,  # 验证同样保持低显存
            gradient_accumulation_steps=8,  # 等效 batch size 仍为 8
            num_train_epochs=3,  # 大模型少训一轮，通常已有较强指令跟随能力
            logging_steps=5,  # 记录训练过程
            eval_strategy="steps",  # 定期验证
            eval_steps=25,  # 验证间隔
            save_steps=25,  # checkpoint 保存间隔
            bf16=True,  # 使用 bf16 训练/推理
        ),
        model_type="causal_lm",  # causal LM 类型
        model_kwargs=common_model_kwargs,  # 共用加载参数
        formatting_func=formatting_function,  # 共用 prompt/completion 格式
        compute_metrics=schema_link_metrics,  # 共用轻量验证指标
        generation_config=common_generation_config,  # 共用确定性生成配置
    ),
    # RFModelConfig(
    #     model_name="Qwen/Qwen2.5-1.5B-Instruct",  # 同一个 1.5B base model，测试稍大学习率
    #     peft_config=peft_configs,  # 与 2 组 LoRA config 组合
    #     training_args=RFSFTConfig(
    #         learning_rate=2e-4,  # 比上一组更激进，用于比较收敛速度和最终效果
    #         lr_scheduler_type="cosine",  # 保持 scheduler 一致
    #         per_device_train_batch_size=1,  # 控制显存
    #         per_device_eval_batch_size=1,  # 控制验证显存
    #         gradient_accumulation_steps=8,  # 保持等效 batch size
    #         num_train_epochs=3,  # 与上一组 1.5B 实验保持 epoch 一致
    #         logging_steps=5,  # 定期记录 loss
    #         eval_strategy="steps",  # 定期验证
    #         eval_steps=25,  # 每 25 步验证
    #         save_steps=25,  # 每 25 步保存 checkpoint
    #         bf16=True,  # 使用 bf16
    #     ),
    #     model_type="causal_lm",  # causal LM 训练类型
    #     model_kwargs=common_model_kwargs,  # 共用模型加载参数
    #     formatting_func=formatting_function,  # 共用样本格式化函数
    #     compute_metrics=schema_link_metrics,  # 共用验证指标
    #     generation_config=common_generation_config,  # 共用生成参数
    # ),
])

In [11]:
experiment = Experiment(
    experiment_name="schema-linking-baseline-qwen-lora",  # RapidFireAI 实验名称；每次大实验建议唯一命名
    mode="fit",  # fit 模式表示执行训练/微调
)

An experiment with the same name already exists. Created a new experiment 'schema-linking-baseline-qwen-lora_2' with Experiment ID: 4 and Metric Experiment ID: 76 at /home/duw003/rapidfireai/rapidfire_experiments/schema-linking-baseline-qwen-lora_2


In [12]:
config_group = RFGridSearch(
    configs=config_set,  # 传入上面定义的多组模型/训练/LoRA 配置
    trainer_type="SFT",  # 指定使用 RapidFireAI 包装的 SFTTrainer
)

In [13]:
experiment.run_fit(
    config_group,  # RapidFireAI grid search 配置组
    create_model,  # 每个 config 启动时调用该函数加载模型和 tokenizer
    train_dataset,  # 训练集 Dataset
    eval_dataset,  # 验证集 Dataset，用于比较 config
    num_chunks=4,  # RapidFireAI 的分块调度粒度，便于多 config 训练管理
    seed=42,  # 固定随机种子，增强实验可复现性
)

INFO 05-24 12:07:12 [__init__.py:216] Automatically detected platform cuda.
Started 1 worker processes successfully
Created workers
INFO 05-24 12:07:25 [__init__.py:216] Automatically detected platform cuda.


In [14]:
experiment.end()  # 显式结束实验，释放/记录 RapidFireAI 实验状态

Experiment schema-linking-baseline-qwen-lora_2 ended
Workers stopped
